In [ ]:
with open('Brothers_Karamazov.txt', 'r', encoding='utf-8') as f:
    karamazov_bros = f.read()

with open('Crime_and_Punishment.txt', 'r', encoding='utf-8') as f:
    raskolnikov = f.read()

text = '\n'.join([karamazov_bros, raskolnikov])

In [ ]:
print(text[-100:])
print(f'{len(text)} dostoyevskyan characters in the dataset')

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

In [ ]:
strtoint = { c:i for i,c in enumerate(chars) }
inttostr = { i:c for i,c in enumerate(chars) }

encode = lambda s: [strtoint[c] for c in s]
decode = lambda li: ''.join(inttostr[i] for i in li)

print(encode('ciao tati'))
print(decode(encode('ciao tati')))

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)

In [ ]:
print(data.shape, data.dtype)
print(data[:100])

In [ ]:
# split into training and validation data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8

print(train_data[:block_size+1])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for i in range(block_size):
    context = x[:i+1]
    target = y[i]
    print(f"When input is {context} the target is: {target}")

In [ ]:
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    random_positions = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in random_positions])
    y = torch.stack([data[i+1:i+1+block_size] for i in random_positions])
    return x, y

In [ ]:
xb, yb = get_batch('train')
print(f"inputs: {xb.shape}\n{xb}") # xb will be the input fed to the transformer
print(f"targets: {yb.shape}\n{yb}")

In [ ]:
# just an example of how xb and yb interact
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"When input is {context} target is: {target}")

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        if targets == None:
            loss=None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)

print(logits.shape, loss)

In [ ]:
def generate_answer(max_new_tokens=100):
    print(decode(model.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens)[0].tolist()))

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward() # checks which directions the weights should go
    optimizer.step() # update the weights

print(loss.item())

In [ ]:
generate_answer(300)